# 02 · Training — Fine-tuning de RoBERTa-base en SQuAD v2

**Modelo:** `FacebookAI/roberta-base` (checkpoint pretrained original, **sin** fine-tuning previo de QA).
**Tarea:** Extractive Question Answering con preguntas no contestables.

Etapas:
1. Configuración y detección de GPU.
2. Tokenización con *sliding window* (`max_seq_length=384`, `doc_stride=128`).
3. Fine-tuning con `Trainer` (3 epochs, LR 3e-5, scheduler lineal, warmup 10%).
4. Guardado del modelo final + curva de pérdida.
5. Evaluación en validation: EM y F1 global, contestables y no contestables.
6. Latencia de inferencia en **CPU**.

**Antes de empezar:** en Colab, `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4 o mejor)`.

**Si Colab se desconecta:** los checkpoints se guardan en Drive cada 1,000 pasos.
Vuelve a ejecutar el notebook desde el inicio y el entrenamiento **continúa desde el último checkpoint**.
Para empezar un experimento nuevo desde cero, borra la carpeta `checkpoints/` en Drive.

Archivos generados en `logs_roberta/`: `loss.png`, `train_results.json`, `eval_results.json`.
Fuera de `logs_roberta/` (solo en Drive): el modelo, los checkpoints y `predictions_val.json`
(lo usa el notebook 03).

## 1. Instalación, rutas y configuración

In [ ]:
# ⚠️ Es normal que pip muestre un aviso de 'dependency conflicts' con gradio/diffusers:
#    este notebook no usa esos paquetes. Tras esta celda, la primera vez:
#    Entorno de ejecución → Reiniciar sesión, y luego Ejecutar todas de nuevo.
# Colab ya trae torch, numpy y matplotlib. Fijamos transformers < 5 para que la API
# (Trainer, pipeline) sea la misma con la que se escribió este notebook.
%pip install -q "transformers>=4.46,<5" "datasets>=3.0" "evaluate>=0.4" "accelerate>=0.34"

In [ ]:
import os, sys

# ── Rutas ─────────────────────────────────────────────────────────────
# En Colab todo se guarda en Google Drive para que no se pierda si la sesión
# se desconecta. Fuera de Colab se usa la raíz del repositorio.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/roberta_squad2"
else:
    BASE_DIR = os.path.abspath("..")   # el notebook vive en pipeline_roberta/

LOGS_DIR   = os.path.join(BASE_DIR, "logs_roberta")                 # resultados para GitHub
MODEL_DIR  = os.path.join(BASE_DIR, "roberta-base-squad2-final")    # modelo final (NO va a GitHub)
CKPT_DIR   = os.path.join(BASE_DIR, "checkpoints")                  # checkpoints intermedios
PRED_FILE  = os.path.join(BASE_DIR, "predictions_val.json")         # predicciones de validación
os.makedirs(LOGS_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("LOGS_DIR:", LOGS_DIR)

In [ ]:
import json, time, glob, platform, collections
import numpy as np
import torch
import matplotlib.pyplot as plt
import transformers
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments,
                          Trainer, TrainerCallback, default_data_collator, set_seed, pipeline)
from transformers.trainer_utils import get_last_checkpoint
import evaluate

CONFIG = {
    "model_checkpoint": "FacebookAI/roberta-base",
    "dataset": "rajpurkar/squad_v2",
    "max_train_samples": None,     # None = train completo. Ej.: 30000 para igualar el subset de ModernBERT
    "num_train_epochs": 3,
    "learning_rate": 3e-5,
    "lr_scheduler_type": "linear",
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_seq_length": 384,
    "doc_stride": 128,
    "effective_batch_size": 32,
    "per_device_train_batch_size": None,   # None = automático según la GPU. Si hay OOM, pon 8.
    "n_best": 20,
    "max_answer_length": 30,
    "null_threshold": 0.0,         # span si score(span) > score(<s>) + threshold
    "seed": 42,
}
set_seed(CONFIG["seed"])

## 2. GPU y tamaño de batch

In [ ]:
assert torch.cuda.is_available(), "No hay GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU."

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
use_bf16   = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere o superior (A100, L4)
use_fp16   = not use_bf16                                  # T4 / V100

bs = CONFIG["per_device_train_batch_size"] or (32 if gpu_mem_gb >= 35 else 16)
assert CONFIG["effective_batch_size"] % bs == 0
grad_accum = CONFIG["effective_batch_size"] // bs
CONFIG["per_device_train_batch_size"] = bs
CONFIG["gradient_accumulation_steps"] = grad_accum
CONFIG["precision"] = "bf16" if use_bf16 else "fp16"

print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
print(f"Batch por dispositivo: {bs} × acumulación {grad_accum} = efectivo {bs * grad_accum}")
print(f"Precisión mixta: {CONFIG['precision']}")

## 3. Datos y tokenización

- **Train:** cada ejemplo se divide en ventanas de 384 tokens. Si la respuesta no cae
  dentro de una ventana (o la pregunta no tiene respuesta), la etiqueta apunta al token
  `<s>` (posición 0), que funciona como "sin respuesta".
- **Validation (pérdida):** misma tokenización que train, para calcular `eval_loss`.
- **Validation (predicciones):** guardamos `offset_mapping` para convertir posiciones
  de tokens en texto del contexto.

In [ ]:
raw = load_dataset(CONFIG["dataset"])
raw_train, raw_val = raw["train"], raw["validation"]
if CONFIG["max_train_samples"]:
    raw_train = raw_train.shuffle(seed=CONFIG["seed"]).select(range(CONFIG["max_train_samples"]))
print(f"Train: {len(raw_train):,} ejemplos | Validation: {len(raw_val):,} ejemplos")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_checkpoint"])
MAX_LEN, STRIDE = CONFIG["max_seq_length"], CONFIG["doc_stride"]

In [ ]:
def tokenize(examples):
    return tokenizer(
        [q.strip() for q in examples["question"]], examples["context"],
        max_length=MAX_LEN, truncation="only_second", stride=STRIDE,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
    )

def prepare_train_features(examples):
    enc = tokenize(examples)
    sample_map = enc.pop("overflow_to_sample_mapping")
    offsets_all = enc.pop("offset_mapping")
    starts, ends = [], []
    for i, offsets in enumerate(offsets_all):
        answer = examples["answers"][sample_map[i]]
        cls_index = enc["input_ids"][i].index(tokenizer.cls_token_id)   # 0 en RoBERTa
        seq_ids = enc.sequence_ids(i)
        if len(answer["answer_start"]) == 0:                           # no contestable
            starts.append(cls_index); ends.append(cls_index); continue
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        ctx_start = seq_ids.index(1)
        ctx_end = len(seq_ids) - 1 - seq_ids[::-1].index(1)
        if offsets[ctx_start][0] > start_char or offsets[ctx_end][1] < end_char:
            starts.append(cls_index); ends.append(cls_index); continue  # respuesta fuera de la ventana
        s = ctx_start
        while s <= ctx_end and offsets[s][0] <= start_char: s += 1
        e = ctx_end
        while e >= ctx_start and offsets[e][1] >= end_char: e -= 1
        starts.append(s - 1); ends.append(e + 1)
    enc["start_positions"], enc["end_positions"] = starts, ends
    return enc

def prepare_val_features(examples):
    enc = tokenize(examples)
    sample_map = enc.pop("overflow_to_sample_mapping")
    enc["example_id"] = []
    for i in range(len(enc["input_ids"])):
        enc["example_id"].append(examples["id"][sample_map[i]])
        seq_ids = enc.sequence_ids(i)
        enc["offset_mapping"][i] = [o if seq_ids[k] == 1 else None
                                    for k, o in enumerate(enc["offset_mapping"][i])]
    return enc

train_ds = raw_train.map(prepare_train_features, batched=True, remove_columns=raw_train.column_names,
                         desc="Tokenizando train")
val_loss_ds = raw_val.map(prepare_train_features, batched=True, remove_columns=raw_val.column_names,
                          desc="Tokenizando validation (loss)")
val_pred_ds = raw_val.map(prepare_val_features, batched=True, remove_columns=raw_val.column_names,
                          desc="Tokenizando validation (predicciones)")
print(f"Features — train: {len(train_ds):,} | val loss: {len(val_loss_ds):,} | val pred: {len(val_pred_ds):,}")

In [ ]:
# Comprobación rápida: la etiqueta de un ejemplo contestable debe reconstruir la respuesta
i = next(k for k in range(len(train_ds)) if train_ds[k]["start_positions"] != 0)
f = train_ds[i]
print("Span reconstruido:", repr(tokenizer.decode(f["input_ids"][f["start_positions"]:f["end_positions"] + 1])))

## 4. Modelo

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(CONFIG["model_checkpoint"])
num_params = model.num_parameters()
print(f"Parámetros: {num_params:,}")

## 5. Entrenamiento

`SessionTimer` registra en Drive cuánto dura cada sesión de entrenamiento, para poder
sumar el tiempo real aunque Colab se desconecte y haya que reanudar.

In [ ]:
SESSIONS_FILE = os.path.join(CKPT_DIR, "training_sessions.json")
os.makedirs(CKPT_DIR, exist_ok=True)
last_ckpt = get_last_checkpoint(CKPT_DIR)
if last_ckpt is None and os.path.exists(SESSIONS_FILE):
    os.remove(SESSIONS_FILE)        # entrenamiento nuevo: no arrastrar sesiones anteriores
print("Reanudando desde:" if last_ckpt else "Entrenamiento nuevo.", last_ckpt or "")

class SessionTimer(TrainerCallback):
    def _write(self, state):
        self.sessions[-1]["seconds"] = round(time.time() - self.t0, 1)
        self.sessions[-1]["last_step"] = state.global_step
        with open(SESSIONS_FILE, "w") as f:
            json.dump(self.sessions, f, indent=2)
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
        self.sessions = json.load(open(SESSIONS_FILE)) if os.path.exists(SESSIONS_FILE) else []
        self.sessions.append({"gpu": gpu_name, "resumed_from": last_ckpt, "seconds": 0})
        self._write(state)
    def on_log(self, args, state, control, **kw):  self._write(state)
    def on_save(self, args, state, control, **kw): self._write(state)
    def on_train_end(self, args, state, control, **kw): self._write(state)

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=CONFIG["num_train_epochs"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=grad_accum,
    fp16=use_fp16, bf16=use_bf16,
    eval_strategy="epoch",
    save_strategy="steps", save_steps=1000, save_total_limit=2,
    logging_steps=100,
    seed=CONFIG["seed"],
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_loss_ds,
    data_collator=default_data_collator, processing_class=tokenizer,
    callbacks=[SessionTimer()],
)

In [ ]:
train_result = trainer.train(resume_from_checkpoint=last_ckpt)

## 6. Guardar modelo final y curva de pérdida

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

weights = glob.glob(os.path.join(MODEL_DIR, "*.safetensors")) + glob.glob(os.path.join(MODEL_DIR, "*.bin"))
weights_bytes = sum(os.path.getsize(p) for p in weights)
dir_bytes = sum(os.path.getsize(os.path.join(MODEL_DIR, f)) for f in os.listdir(MODEL_DIR))
print(f"Modelo guardado en {MODEL_DIR}")
print(f"Pesos: {weights_bytes/1024**2:.1f} MB | Carpeta completa: {dir_bytes/1024**2:.1f} MB")

sessions = json.load(open(SESSIONS_FILE))
train_seconds = sum(s["seconds"] for s in sessions)
print(f"Tiempo total de entrenamiento: {train_seconds/3600:.2f} h en {len(sessions)} sesión(es)")

In [ ]:
hist = trainer.state.log_history          # incluye lo registrado antes de reanudar
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h and "eval_loss" not in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

plt.figure(figsize=(11, 5))
plt.plot(*zip(*tr), label="Train loss", color="#1565C0", alpha=0.85)
if ev:
    plt.plot(*zip(*ev), "o--", label="Validation loss (por epoch)", color="#C62828", lw=2)
steps_per_epoch = trainer.state.max_steps / CONFIG["num_train_epochs"]
for e in range(1, CONFIG["num_train_epochs"]):
    plt.axvline(e * steps_per_epoch, color="gray", ls=":", lw=1)
plt.xlabel("Paso de optimización"); plt.ylabel("Loss")
plt.title(f"RoBERTa-base / SQuAD v2 — LR={CONFIG['learning_rate']}, epochs={CONFIG['num_train_epochs']}, "
          f"batch efectivo={bs*grad_accum}")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, "loss.png"), dpi=150)
plt.show()

## 7. Evaluación en validation (11,873 preguntas)

Para cada pregunta se combinan todas sus ventanas:
- **Mejor span:** máximo de `start_logit + end_logit` entre los `n_best` candidatos válidos.
- **Score de "sin respuesta":** `start_logit[<s>] + end_logit[<s>]` (mínimo entre ventanas).
- Se predice el span si `score_span > score_nulo + threshold`; si no, la respuesta es `""`.

La métrica oficial `squad_v2` cuenta como **correcta** una predicción vacía `""` en una
pregunta no contestable (EM = F1 = 1 para ese ejemplo).

In [ ]:
def postprocess(raw_examples, features, start_logits, end_logits, n_best=20, max_answer_length=30):
    feats_per_example = collections.defaultdict(list)
    for i, ex_id in enumerate(features["example_id"]):
        feats_per_example[ex_id].append(i)
    offsets_all = list(features["offset_mapping"])
    results = []
    for ex in raw_examples:
        best_text, best_score, min_null = "", -1e9, None
        for fi in feats_per_example[ex["id"]]:
            sl, el, offsets = start_logits[fi], end_logits[fi], offsets_all[fi]
            null_score = float(sl[0] + el[0])            # posición 0 = <s>
            min_null = null_score if min_null is None else min(min_null, null_score)
            for s in np.argsort(sl)[-1:-n_best - 1:-1]:
                for e in np.argsort(el)[-1:-n_best - 1:-1]:
                    if offsets[s] is None or offsets[e] is None: continue
                    if e < s or e - s + 1 > max_answer_length: continue
                    score = float(sl[s] + el[e])
                    if score > best_score:
                        best_score, best_text = score, ex["context"][offsets[s][0]:offsets[e][1]]
        results.append({"id": ex["id"], "best_span_text": best_text, "best_span_score": best_score,
                        "null_score": min_null, "score_diff": best_score - min_null})
    return results

def apply_threshold(results, threshold):
    return {r["id"]: (r["best_span_text"] if r["score_diff"] > threshold else "") for r in results}

squad_metric = evaluate.load("squad_v2")
def squad_v2_scores(pred_dict, raw_examples):
    preds = [{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in pred_dict.items()]
    refs = [{"id": ex["id"], "answers": ex["answers"]} for ex in raw_examples]
    return squad_metric.compute(predictions=preds, references=refs)

In [ ]:
eval_loss_metrics = trainer.evaluate()          # eval_loss del modelo final

t0 = time.time()
pred_out = trainer.predict(val_pred_ds.remove_columns(["example_id", "offset_mapping"]))
start_logits, end_logits = pred_out.predictions
results = postprocess(raw_val, val_pred_ds, start_logits, end_logits,
                      CONFIG["n_best"], CONFIG["max_answer_length"])
gpu_eval_seconds = time.time() - t0

scores = squad_v2_scores(apply_threshold(results, CONFIG["null_threshold"]), raw_val)
print(f"EM global {scores['exact']:.2f} | F1 global {scores['f1']:.2f}")
print(f"Contestables    EM {scores['HasAns_exact']:.2f} | F1 {scores['HasAns_f1']:.2f}  (n={scores['HasAns_total']})")
print(f"No contestables EM {scores['NoAns_exact']:.2f} | F1 {scores['NoAns_f1']:.2f}  (n={scores['NoAns_total']})")

with open(PRED_FILE, "w", encoding="utf-8") as f:     # lo usa el notebook 03
    json.dump(results, f, ensure_ascii=False)

## 8. Latencia de inferencia en CPU

Se carga el modelo guardado **en CPU** y se mide el tiempo de una pregunta completa
(tokenización + ventanas + modelo + postproceso) con el `pipeline` de Transformers.

⚠️ La CPU de Colab no es la misma que la de tu laptop. Para el reporte final conviene
repetir esta medición en la máquina donde corra la API.

In [ ]:
cpu_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_DIR)   # se carga en CPU
qa_cpu = pipeline("question-answering", model=cpu_model, tokenizer=tokenizer, device=-1)
qa_kwargs = dict(max_seq_len=MAX_LEN, doc_stride=STRIDE, max_answer_len=CONFIG["max_answer_length"],
                 handle_impossible_answer=True)

def time_query(ex):
    t = time.perf_counter()
    qa_cpu(question=ex["question"], context=ex["context"], **qa_kwargs)
    return (time.perf_counter() - t) * 1000

sample = raw_val.shuffle(seed=CONFIG["seed"]).select(range(33))
for ex in sample.select(range(3)): time_query(ex)          # calentamiento
lat = np.array([time_query(ex) for ex in sample.select(range(3, 33))])

longest = raw_val[int(np.argmax([len(c) for c in raw_val["context"]]))]
lat_long = np.array([time_query(longest) for _ in range(5)])

cpu_latency = {
    "device": "cpu",
    "cpu": platform.processor() or platform.machine(),
    "cpu_count": os.cpu_count(),
    "torch_threads": torch.get_num_threads(),
    "environment": "Google Colab" if IN_COLAB else "local",
    "method": "transformers pipeline, 30 preguntas aleatorias de validation (3 de calentamiento)",
    "mean_ms": round(float(lat.mean()), 1),
    "p50_ms": round(float(np.percentile(lat, 50)), 1),
    "p95_ms": round(float(np.percentile(lat, 95)), 1),
    "long_context_chars": len(longest["context"]),
    "long_context_mean_ms": round(float(lat_long.mean()), 1),
}
print(json.dumps(cpu_latency, indent=2))

## 9. Guardar `train_results.json` y `eval_results.json`

In [ ]:
train_results = {
    "model_checkpoint": CONFIG["model_checkpoint"],
    "config": CONFIG,
    "data": {
        "train_examples": len(raw_train), "train_features": len(train_ds),
        "val_examples": len(raw_val), "val_features": len(val_pred_ds),
    },
    "hardware": {"gpu": gpu_name, "gpu_memory_gb": round(gpu_mem_gb, 1), "precision": CONFIG["precision"]},
    "num_parameters": num_params,
    "model_weights_mb": round(weights_bytes / 1024**2, 1),
    "model_dir_mb": round(dir_bytes / 1024**2, 1),
    "training_time": {
        "total_seconds": round(train_seconds, 1),
        "total_hours": round(train_seconds / 3600, 2),
        "num_sessions": len(sessions),
        "sessions": sessions,
        "note": "Suma de todas las sesiones (incluye la evaluación de loss al final de cada epoch).",
    },
    "global_steps": trainer.state.global_step,
    "final_train_loss_logged": tr[-1][1] if tr else None,
    "trainer_metrics_last_session": train_result.metrics,
    "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
}

eval_results = {
    "split": "validation",
    "num_examples": len(raw_val),
    "null_threshold": CONFIG["null_threshold"],
    "overall":      {"exact_match": scores["exact"],        "f1": scores["f1"],        "count": scores["total"]},
    "answerable":   {"exact_match": scores["HasAns_exact"], "f1": scores["HasAns_f1"], "count": scores["HasAns_total"]},
    "unanswerable": {"exact_match": scores["NoAns_exact"],  "f1": scores["NoAns_f1"],  "count": scores["NoAns_total"]},
    "eval_loss": eval_loss_metrics.get("eval_loss"),
    "gpu_eval_seconds": round(gpu_eval_seconds, 1),
    "cpu_latency": cpu_latency,
    "metric": "evaluate.load('squad_v2'); '' en una pregunta no contestable cuenta como correcto",
}

for name, obj in [("train_results.json", train_results), ("eval_results.json", eval_results)]:
    with open(os.path.join(LOGS_DIR, name), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    print("✅ Guardado:", os.path.join(LOGS_DIR, name))

In [ ]:
print("=" * 60)
print("RESUMEN — RoBERTa-base / SQuAD v2")
print("=" * 60)
print(f"Train: {len(raw_train):,} ejemplos ({len(train_ds):,} features) | Val: {len(raw_val):,}")
print(f"Epochs {CONFIG['num_train_epochs']} | LR {CONFIG['learning_rate']} {CONFIG['lr_scheduler_type']} "
      f"warmup {CONFIG['warmup_ratio']:.0%} | batch {bs}×{grad_accum}={bs*grad_accum}")
print(f"GPU: {gpu_name} ({CONFIG['precision']}) | Tiempo: {train_seconds/3600:.2f} h")
print(f"Parámetros: {num_params:,} | Pesos: {weights_bytes/1024**2:.0f} MB")
print(f"EM {scores['exact']:.2f} / F1 {scores['f1']:.2f} "
      f"(HasAns {scores['HasAns_f1']:.2f} F1, NoAns {scores['NoAns_f1']:.2f} F1)")
print(f"Latencia CPU media: {cpu_latency['mean_ms']:.0f} ms")

**Siguiente paso:** `03_error_analysis_roberta.ipynb` (no necesita GPU).